<a href="https://colab.research.google.com/github/acm-research-f26/Circuitree/blob/Jotish-Chowdary-Boddu/fgnn2_eval/FGNN2_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
  from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/ZeayW/FGNN2.git
!pip install -q uv
!uv venv /content/fgnn_env --python 3.12 --seed
!/content/fgnn_env/bin/pip install -q "numpy<2" torch==2.4.0 --index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://pypi.org/simple
!/content/fgnn_env/bin/pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
!mkdir -p FGNN2/checkpoints/example
!gdown -q 1NGAJRedx040A4EeN9mALXsly87TRseRF -O FGNN2/checkpoints/example/weight.pth
!unzip -q -o "/content/drive/MyDrive/Circuitree/variants.zip" -d /content/raw_variants

Mounted at /content/drive
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment with seed packages at: fgnn_env
 + pip==26.3
Activate with: source fgnn_env/bin/activate


In [2]:
!find /content/raw_variants -type f | wc -l
!find /content/raw_variants -type f | sort | head -15

101
/content/raw_variants/c1355_v10.bench
/content/raw_variants/c1355_v1.bench
/content/raw_variants/c1355_v2.bench
/content/raw_variants/c1355_v3.bench
/content/raw_variants/c1355_v4.bench
/content/raw_variants/c1355_v5.bench
/content/raw_variants/c1355_v6.bench
/content/raw_variants/c1355_v7.bench
/content/raw_variants/c1355_v8.bench
/content/raw_variants/c1355_v9.bench
/content/raw_variants/c1908_v10.bench
/content/raw_variants/c1908_v1.bench
/content/raw_variants/c1908_v2.bench
/content/raw_variants/c1908_v3.bench
/content/raw_variants/c1908_v4.bench


In [3]:
!find /content/raw_variants -name "*.bench" | sort | head -1 | xargs head -20

# Benchmark "c1355" written by ABC on Sun Oct  4 19:55:48 2026
INPUT(1)
INPUT(8)
INPUT(15)
INPUT(22)
INPUT(29)
INPUT(36)
INPUT(43)
INPUT(50)
INPUT(57)
INPUT(64)
INPUT(71)
INPUT(78)
INPUT(85)
INPUT(92)
INPUT(99)
INPUT(106)
INPUT(113)
INPUT(120)
INPUT(127)


In [2]:
import os, shutil, glob

src = '/content/raw_variants'
dst = '/content/iscas_variants'

for path in glob.glob(src + '/*.bench'):
    name = os.path.basename(path)
    circuit = name.split('_v')[0]
    os.makedirs(dst + '/' + circuit, exist_ok=True)
    shutil.copy(path, dst + '/' + circuit + '/' + name)

for circuit in sorted(os.listdir(dst)):
    count = len(os.listdir(dst + '/' + circuit))
    print(circuit, '->', count, 'files')

c1355 -> 10 files
c1908 -> 10 files
c2670 -> 10 files
c3540 -> 10 files
c432 -> 10 files
c499 -> 10 files
c5315 -> 10 files
c6288 -> 10 files
c7552 -> 10 files
c880 -> 10 files


In [5]:
!cat /content/raw_variants/*.bench | grep "=" | sed 's/.*= *\([A-Za-z]*\).*/\1/' | sort | uniq -c

  81933 AND
    854 BUFF
  75751 NOT
     10 vdd


In [4]:
!cd /content/FGNN2/src && /content/fgnn_env/bin/python eval_retrieval.py --bench_root /content/iscas_variants --weights ../checkpoints/example/weight.pth --flag_inv --ks 1 5 10

DGL backend not selected or invalid.  Assuming PyTorch for now.
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
/content/FGNN2/src/eval_retrieval.py:177: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setti

In [5]:
!cd /content/FGNN2/src && /content/fgnn_env/bin/python eval_retrieval.py --bench_root /content/iscas_variants --flag_inv --ks 1 5 10

parsed 100 circuits from 10 groups
                       hit@1: 0.9200
                       hit@5: 0.9900
                      hit@10: 1.0000
                         MRR: 0.9521
                    top1_acc: 0.9200
                   mean_rank: 1.1700
                 num_queries: 100
                num_circuits: 100
                      device: cuda:0
                time_parse_s: 1.0307
                time_build_s: 0.0668
                time_embed_s: 0.5715
        ms_per_circuit_embed: 5.7147
   time_search_all_queries_s: 0.0020
               avg_and_nodes: 819.3300


In [8]:
!cd /content/FGNN2/src && /content/fgnn_env/bin/python eval_retrieval.py --bench_root /content/iscas_variants --weights ../checkpoints/example/weight.pth --flag_inv --ks 1 5 10 --show_misses

parsed 100 circuits from 10 groups
                       hit@1: 0.9600
                       hit@5: 0.9900
                      hit@10: 1.0000
                         MRR: 0.9744
                    top1_acc: 0.9600
                   mean_rank: 1.1200
                 num_queries: 100
                num_circuits: 100
                      device: cuda:0
                time_parse_s: 0.6557
                time_build_s: 0.0635
                time_embed_s: 0.5720
        ms_per_circuit_embed: 5.7197
   time_search_all_queries_s: 0.0021
               avg_and_nodes: 819.3300

MISSES (circuit -> its #1 pick, which was NOT a twin):
        c1355_v1.bench -> c499_v2.bench        similarity 0.9975
        c1355_v7.bench -> c499_v9.bench        similarity 0.9982
        c1355_v8.bench -> c499_v10.bench       similarity 0.9948
         c499_v9.bench -> c1355_v7.bench       similarity 0.9982
